# Setting Up All Artifacts details

In [1]:
## Give appropriate permission to the directory "FOLDER_WITH_ARTIFACTS" you are working with
import os
# "/local/mnt/workspace/snpe/2.24.0.240626"
os.environ['SNPE_ROOT']="/opt/qcom/aistack/qairt/2.25.0.240728"#set up your snpe path here.
os.environ['RAW_FILE_FOLDER']="raw"
os.environ['DLC32']="models/yolo_nas_fp32.dlc"
os.environ['DLC8']="models/yolo_nas_w8a8.dlc"
os.environ['TARGET_INPUT_LIST']="input.txt"
os.environ['ONDEVICE_FOLDER']="yolonas"
os.environ['DEVICE_HOST']="192.168.31.139"
os.environ['DEVICE_ID']="503bd507" #change with your device-id. Use command "adb devices" to get devices names.
os.environ['SNPE_TARGET_ARCH']="aarch64-android"
os.environ['SNPE_TARGET_STL']="libc++_shared.so"

In [2]:
## Note- Use python3.8 or above for generating onnx
!pip install super-gradients==3.1.2
import torch
from super_gradients.training import models
from super_gradients.common.object_names import Models
import cv2
import numpy as np
import os

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.aliyun.com/pypi/simple
The console stream is logged into /home/liuqi/sg_logs/console.log


[2025-03-02 17:46:15] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it
[2025-03-02 17:46:20] WARNING - __init__.py - Failed to import pytorch_quantization
[2025-03-02 17:46:20] WARNING - calibrator.py - Failed to import pytorch_quantization
[2025-03-02 17:46:20] WARNING - export.py - Failed to import pytorch_quantization
[2025-03-02 17:46:20] WARNING - selective_quantization_utils.py - Failed to import pytorch_quantization


## Getting The dataset
Please, fill coco dataset link in below code block.

In [3]:
# !wget https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip -q --show-progress
# !wget <coco_dataset_link> -q --show-progress
# !unzip val2017.zip
# !unzip coco2017labels.zip
# !mkdir "raw"

In [4]:
# files = os.listdir('val2017') #val2017 is the datatset folder path. Keeping only 15 images.
# for file in files[15:]:
#     os.remove("val2017/"+file)

In [5]:
# %%bash
# rm -rf coco
# rm -rf coco2017labels.zip
# rm -rf val2017.zip

## Getting the ONNX Model

In [6]:
os.makedirs('models', exist_ok=True)

In [7]:
# proxy = 'http://192.168.31.241:7897'
# os.environ['http_proxy'] = proxy
# os.environ['HTTP_PROXY'] = proxy
# os.environ['https_proxy'] = proxy
# os.environ['HTTPS_PROXY'] = proxy

In [8]:
model = models.get(Models.YOLO_NAS_S, pretrained_weights="coco")
# Prpare model for conversion
# Input size is in format of [Batch x Channels x Width x Height] where 640 is the standard COCO dataset dimensions
model.eval()
model.prep_model_for_conversion(input_size=[1, 3, 320, 320])
# Create dummy_input
dummy_input = torch.randn([1, 3, 320, 320], device="cpu")
# Convert model to onnx
torch.onnx.export(model, dummy_input, "models/yolo_nas_s.onnx", opset_version=11)

[2025-03-02 17:46:20] INFO - checkpoint_utils.py - License Notification: YOLO-NAS pre-trained weights are subjected to the specific license terms and conditions detailed in 
https://github.com/Deci-AI/super-gradients/blob/master/LICENSE.YOLONAS.md
By downloading the pre-trained weight files you agree to comply with these terms.


#### Getting the FP32 Model

In [9]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-onnx-to-dlc -i models/yolo_nas_s.onnx -o app/src/main/assets/yolo_nas_s.dlc --out_node 885 --out_node 893

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728


2025-03-02 17:46:46,150 - 235 - INFO - Simplified model validation is successful
2025-03-02 17:46:49,073 - 235 - INFO - user_provided_output is same as graph_output..skipping update_output_names call
2025-03-02 17:46:51,995 - 235 - INFO - INFO_INITIALIZATION_SUCCESS: 
2025-03-02 17:46:52,313 - 235 - INFO - INFO_CONVERSION_SUCCESS: Conversion completed successfully


2025-03-02 17:46:55,017 - 235 - INFO - INFO_WRITE_SUCCESS: 


## Preprocessing

In [10]:
def preprocess(original_image):
    resized_image = cv2.resize(original_image, (320, 320))
    resized_image = resized_image/255
    return resized_image
##Please download Coco2014 dataset and give the path here
dataset_path = "val2017/"
!mkdir -p raw
filenames=[]
for path in os.listdir(dataset_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dataset_path, path)):
        filenames.append(os.path.join(dataset_path, path))
for filename in filenames:
    original_image = cv2.imread(filename)
    img = preprocess(original_image)
    img = img.astype(np.float32)
    img.tofile("raw/"+filename.split("/")[-1].split(".")[0]+".raw")

In [11]:
%%bash
find raw -name *.raw > input.txt

## Quantize the DLC

In [ ]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-quantize --input_dlc app/src/main/assets/yolo_nas_s.dlc --input_list input.txt --use_enhanced_quantizer --use_adjusted_weights_quantizer --axis_quant --output_dlc app/src/main/assets/Quant_intermediate_yoloNas_s_320.dlc

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728


[INFO] InitializeStderr: DebugLog initialized.
[WARNING] --axis_quant is deprecated, use --use_per_channel_quantization option.
[WARNING] --use_enhanced_quantizer option is deprecated, use --param_quantizer and --act_quantizer options.
[WARNING] --use_adjusted_weights_quantizer option is deprecated, use --param_quantizer option.
[INFO] Processed command-line arguments


IrQuantizer: Quantizer param type: adjusted will be deprecated in future releases
IrQuantizer: Quantizer type: adjusted is no longer supported. Using TF quantizer instead


[INFO] Quantized parameters


    25.6ms [  INFO ] Inferences will run in sync mode
    39.8ms [  INFO ] Initializing logging in the backend. Callback: [0x55ea817fab60], Log Level: [3]
    39.8ms [  INFO ] No BackendExtensions lib provided;initializing NetRunBackend Interface
    15.4ms [  INFO ] [QNN_CPU] CpuBackend creation start
    15.4ms [  INFO ] [QNN_CPU] CpuBackend creation end
    55.2ms [WARNING] Unable to find a device with NetRunDeviceKeyDefault in Library NetRunBackendLibKeyDefault
    55.2ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
    26.5ms [  INFO ] [QNN_CPU] QnnContext create start
    26.5ms [  INFO ] [QNN_CPU] QnnContext create end
    66.6ms [  INFO ] Entering QuantizeRuntimeApp flow
    66.7ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
    34.5ms [  INFO ] [QNN_CPU] CpuGraph creation start
    34.5ms [  INFO ] [QNN_CPU] CpuGraph creation end
    34.5ms [  INFO ] [QNN_CPU] QnnGraph create end
   240.7ms [  INFO ] [QNN

[INFO] Generated activations


[INFO] Saved quantized dlc to: app/src/main/assets/Quant_intermediate_yoloNas_s_320.dlc
[INFO] DebugLog shutting down.


 [  INFO ] [QNN_CPU] QnnGraph execute start
 29993.1ms [  INFO ] [QNN_CPU] QnnGraph execute end
 29958.8ms [  INFO ] cleaning up resources for input tensors
 29958.8ms [  INFO ] cleaning up resources for output tensors
 30072.2ms [  INFO ] [QNN_CPU] QnnGraph execute start
 31122.1ms [  INFO ] [QNN_CPU] QnnGraph execute end
 31087.8ms [  INFO ] cleaning up resources for input tensors
 31087.8ms [  INFO ] cleaning up resources for output tensors
 31197.5ms [  INFO ] [QNN_CPU] QnnGraph execute start
 32157.9ms [  INFO ] [QNN_CPU] QnnGraph execute end
 32123.7ms [  INFO ] cleaning up resources for input tensors
 32123.7ms [  INFO ] cleaning up resources for output tensors
 32231.8ms [  INFO ] [QNN_CPU] QnnGraph execute start
 32932.6ms [  INFO ] [QNN_CPU] QnnGraph execute end
 32898.3ms [  INFO ] cleaning up resources for input tensors
 32898.3ms [  INFO ] cleaning up resources for output tensors
 33017.9ms [  INFO ] [QNN_CPU] QnnGraph execute start
 33771.5ms [  INFO ] [QNN_CPU] QnnGraph 

<b>- Based on the device where you will execute the model set --htp_socs to sm8650 or sm8550</b>

In [13]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-graph-prepare --input_dlc models/yolo_nas_w8a8.dlc --set_output_tensors=885,893 --htp_socs=sm8650 --output_dlc=models/Quant_yoloNas_s_320.dlc

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /mnt/d/media/code/opt/qcom/aistack/qairt/2.25.0.240728


[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM8650
[USER_INFO] Target device backend record identifier: HTP_V75_SM8650_8MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V75_SM8650_8MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x5562f1c359d0
[INFO] Found Interface Provider (v2.18)
[USER_WARNING] QnnDsp <W> Initializing HtpProvider
[USER_WARNING] QnnDsp <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING] QnnDsp <W> Performance Estimates unsupported
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x5562f19da528
[USER_INFO] Offline Prepare VTCM size(MB) selected = 8
[USER_INFO] Offline Prepare Optimization Level passed = 2
[USER_WARNING] QnnDsp <W> Output padding param cannot be set explicitly.

[USER_INFO] BackendTerminate triggered
[INFO] DebugLog shutting down.
